# Notebook 1 — Exploratory Data Analysis

Obiettivo: comprendere le distribuzioni dei sensori, la durata del ciclo di vita dei motori e identificare i sensori a varianza costante da rimuovere prima del training.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
# sys.path.append necessario per importare il modulo src/ dall'interno della cartella notebooks/
sys.path.append('..')
from src.preprocessing import load_raw, COLUMNS, DROP_SENSORS

sns.set_style('whitegrid')
DATA_DIR = '../data/raw'

## 1. Caricamento dei dataset

CMAPSS contiene 4 sotto-dataset con diversa complessità:
- **FD001/FD003**: 1 condizione operativa → normalizzazione MinMax globale
- **FD002/FD004**: 6 condizioni operative → normalizzazione cluster-based per regime

Ogni file ha 26 colonne: ID motore, ciclo operativo, **3 impostazioni operative** e 21 sensori.

Le **impostazioni operative** (setting1, setting2, setting3) descrivono le condizioni di volo:
- `setting1` → quota di volo (ft)
- `setting2` → numero di Mach
- `setting3` → angolo del throttle (TRA, %)

In FD001/FD003 questi valori sono quasi costanti (1 regime di volo). In FD002/FD004 variano tra i cicli (6 profili di volo diversi).

Carico solo i file di **train**: i file di test non contengono le etichette RUL (fornite separatamente in `RUL_FDxxx.txt`).

In [ ]:
# Carico solo il train: il test non ha etichette RUL interne
datasets = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    datasets[fd] = load_raw(f'{DATA_DIR}/train_{fd}.txt')
    print(f'{fd}: {datasets[fd].shape}')

## 2. Distribuzione del ciclo di vita dei motori

Per ogni motore calcolo il numero massimo di cicli operativi prima del guasto.
Questa analisi motiva la scelta del **cap a 125 cicli** nella RUL:
poiché la maggior parte dei motori vive molto più di 125 cicli, nella fase iniziale
il degrado non è ancora rilevabile dai sensori — trattare quella fase come "RUL = 125"
è equivalente a dire "non sappiamo ancora quanto mancherà, ma è molto".
Questo è il **piecewise linear RUL**, standard nella letteratura CMAPSS.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (fd, df) in enumerate(datasets.items()):
    # max cycle per motore = durata vita effettiva (cicli totali fino al guasto)
    life = df.groupby('engine_id')['cycle'].max()
    axes[i].hist(life, bins=20, edgecolor='black')
    axes[i].set_title(fd)
    axes[i].set_xlabel('Max cycle')
# FD003 e FD004 mostrano code più lunghe → alta variabilità inter-motore
plt.suptitle('Engine lifecycle distribution')
plt.tight_layout()
plt.savefig('../plots/01_lifecycle_distribution.png', dpi=150)
plt.show()

### Cosa mostra questo grafico

Ogni istogramma rappresenta quanti motori hanno raggiunto una certa durata (in cicli) prima del guasto.

**FD001 e FD002** mostrano distribuzioni concentrate attorno a 150–200 cicli con una coda destra moderata fino a ~375 cicli. La forma è relativamente simmetrica: la variabilità tra motori è contenuta.

**FD003 e FD004** hanno distribuzioni molto più larghe, con motori che arrivano fino a 500+ cicli. La coda destra pronunciata indica un'alta variabilità inter-motore: alcuni motori durano più del doppio della media.

**Implicazione per il cap a 125**: dato che quasi tutti i motori vivono ben oltre 125 cicli, nella fase iniziale i sensori non mostrano ancora variazioni sistematiche legate al degrado. Usare la RUL reale (es. 350 cicli) renderebbe il task di regressione inutilmente difficile in quella fase. Cappare a 125 significa: "oltre una certa soglia, trattali tutti allo stesso modo — non sappiamo ancora distinguerli".

## 3. Selezione dei sensori: analisi della varianza

Calcolo la varianza di ogni sensore per identificare quelli che non cambiano mai durante la vita del motore. Un sensore con varianza zero è costante → non può portare alcuna informazione sul degrado.

**Attenzione**: varianza bassa non significa automaticamente sensore inutile. Sensori come `s8`, `s13`, `s15` hanno varianza piccola ma mostrano comunque un trend correlato al degrado, visibile nell'analisi temporale successiva. La selezione finale (`DROP_SENSORS`) segue la scelta standard della letteratura (Zheng et al., 2017), confermata visivamente dai trend.

In [ ]:
df = datasets['FD001']
sensor_cols = [c for c in df.columns if c.startswith('s')]
# Calcolo varianza di ogni sensore su tutto il dataset e ordino in senso crescente
variance = df[sensor_cols].var().sort_values()

print('Varianza di tutti i sensori (ordinata dal più basso):')
print(variance.round(5))
print()
# I sensori rimossi seguono la scelta della letteratura (Zheng et al., 2017):
# sono quelli con varianza esattamente o quasi zero su tutti i dataset CMAPSS
print('Sensori rimossi (DROP_SENSORS):', DROP_SENSORS)
print('Sensori mantenuti nonostante varianza bassa (s8, s13, s15): mostrano trend di degrado')

### Cosa mostrano questi risultati

I sensori `s1, s5, s6, s10, s16, s18, s19` hanno varianza **esattamente zero**: il loro valore non cambia mai, per nessun motore, in nessun ciclo. Sono sensori che misurano grandezze fisiche non influenzate dal tipo di guasto simulato in CMAPSS e vengono rimossi automaticamente in `load_raw()` prima di qualsiasi elaborazione.

Compaiono anche i **settings** (setting1, setting2, setting3) con varianza quasi nulla: in FD001 c'è una sola condizione operativa, quindi quota, Mach e throttle variano pochissimo tra i cicli. Vengono comunque mantenuti perché in FD002/FD004 (6 condizioni) la loro varianza è alta e sono fondamentali per identificare il regime operativo tramite KMeans.

**Perché s8, s13, s15 rimangono?** La loro varianza è bassa ma non zero: variano di poco in valore assoluto, ma quel piccolo segnale è correlato al processo di degrado. Rimuoverli peggiora le prestazioni — questo è un risultato documentato in letteratura e confermato anche visivamente nel grafico dei trend che segue.

## 4. Andamento dei sensori nel tempo (Motore 1, FD001)

Visualizzo l'evoluzione temporale dei 14 sensori mantenuti per un singolo motore.
Visualizzo 1 motore per leggibilità: il trend è rappresentativo della popolazione.

Sensori chiave da osservare:
- **s2, s3, s4**: trend crescente monotono → degradano con il motore, i più informativi
- **s12, s13**: trend opposti (pressione vs temperatura che si spostano in direzioni diverse)
- **setting3**: piatto → non porta info sul degrado ma utile per la clusterizzazione in FD002/FD004

In [ ]:
# Filtro il motore 1 per l'analisi visiva
engine = df[df['engine_id'] == 1]
# Escludo i sensori costanti già identificati
keep_sensors = [c for c in sensor_cols if c not in DROP_SENSORS]

fig, axes = plt.subplots(4, 4, figsize=(18, 12))
for ax, s in zip(axes.flatten(), keep_sensors):
    ax.plot(engine['cycle'], engine[s])
    ax.set_title(s)
    ax.set_xlabel('Cycle')
plt.suptitle('Sensor trends — Engine 1, FD001')
plt.tight_layout()
plt.savefig('../plots/01_sensor_trends.png', dpi=150)
plt.show()

### Cosa mostra questo grafico

Ogni pannello mostra l'evoluzione di un sensore nel tempo per il motore 1 di FD001 (~200 cicli di vita).

**Sensori con trend di degrado chiaro** (i più utili per la predizione):
- `s2`, `s3`, `s4`: crescita monotona verso fine vita. Misurano grandezze che aumentano progressivamente con il deterioramento del motore (es. temperature, pressioni di scarico)
- `s11`, `s12`: cambiano regime nella parte centrale della vita, mostrando un punto di flessione che segnala l'inizio del degrado accelerato
- `s13`, `s14`, `s15`: trend decrescenti — confermano che `s15`, pur con varianza bassa, **ha un segnale utile**: scende lentamente ma in modo sistematico verso fine vita

**Sensori rumorosi ma informativi** (`s7`, `s8`, `s20`, `s21`): alta variabilità ciclo per ciclo, ma con un trend sottostante debole che emerge con la finestra temporale di 30 cicli.

**I settings in FD001**: `setting1` e `setting2` sono rumore puro (variano casualmente per la singola condizione operativa), `setting3` è piatto. In FD002/FD004 i settings invece assumono valori distinti e discreti corrispondenti ai 6 profili di volo — sono il criterio principale per il clustering.

**Implicazione per la sliding window**: un singolo timestep non è sufficiente per catturare i trend, specialmente nei sensori rumorosi. Una finestra di 30 cicli permette alla rete di vedere abbastanza contesto temporale da distinguere il rumore dal trend sistematico di degrado.